In [1]:
%cd ../../../

/Users/hoangle/Projects/fwo_models


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import joblib
import numpy as np
import polars as pl
from scipy.spatial import distance
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler

# Load dims and raw data

## dim `meal_types`

In [3]:
meals = pl.read_excel("data/processed/phase_4/dim_meals.xlsx")

meal_types = meals.select('meal_id', pl.col('meal_type_1').alias('meal_type'))
meal_types.head()

meal_id,meal_type
i64,str
9017,"""vegan"""
7201,"""vegan"""
9032,"""vegan"""
9102,"""vegan"""
7010,"""vegetarian"""


## dim `meal_names`

In [4]:
meal_names = pl.read_excel("data/processed/phase_4/dim_meal_names.xlsx")
meal_names.head()

meal_id,meal
i64,str
9017,"""""Butter"" härkäpapua & pähkinää"""
7201,"""2023 Härkäpu-sienilasagnette"""
9032,"""Appelisiini-luomukikhernecurry…"
9102,"""Artisokkavugetteja & tuoretoma…"
7010,"""Aurajuusto-pinaattilasagnette"""


## dim `opentime`

In [5]:
dim_opentime = (
    pl
    .from_records([
        {'restaurant': 'che', 'time_open': '10:30', 'time_close': '15:00'},
        {'restaurant': 'exa', 'time_open': '11:00', 'time_close': '14:00'},
        {'restaurant': 'phy', 'time_open': '10:00', 'time_close': '15:00'},
        {'restaurant': 'vik', 'time_open': '10:30', 'time_close': '14:00'},
    ])
    .select(
        'restaurant',
        pl.col('time_open').str.to_datetime("%H:%M"),
        pl.col('time_close').str.to_datetime("%H:%M")
    )
    .with_columns(
        ((pl.col('time_close') - pl.col('time_open')).dt.total_minutes() / 60.).alias('working_duration')
    )
)

dim_opentime.head()

restaurant,time_open,time_close,working_duration
str,datetime[μs],datetime[μs],f64
"""che""",0001-01-01 10:30:00,0001-01-01 15:00:00,4.5
"""exa""",0001-01-01 11:00:00,0001-01-01 14:00:00,3.0
"""phy""",0001-01-01 10:00:00,0001-01-01 15:00:00,5.0
"""vik""",0001-01-01 10:30:00,0001-01-01 14:00:00,3.5


## dim meal names' embedding

In [6]:
meal_embds = pl.read_parquet("data/inter/meal_names_embds.parquet").drop('__index_level_0__', 'meal')
meal_embds.head()

meal_id,embedding
i64,list[f64]
7010,"[0.011746, -0.038603, … 0.036116]"
7010,"[0.013336, -0.046704, … 0.038734]"
1751,"[0.049921, 0.038431, … 0.025644]"
1751,"[0.065358, 0.038125, … 0.022724]"
200006,"[0.021538, 0.027582, … 0.028225]"


In [7]:
meal_embds = meal_embds.group_by('meal_id').first()

In [8]:
meal_sims = meal_embds.join(meal_embds, how='cross')

embds_1 = meal_sims.get_column('embedding').to_numpy()
embds_2 = meal_sims.get_column('embedding_right').to_numpy()
dist = [distance.cosine(e1, e2) for e1, e2 in zip(embds_1, embds_2)]

meal_sims = (
    meal_sims
    .select(
        'meal_id',
        pl.col('meal_id_right').alias('meal_id_sim'),
        pl.Series(dist).alias('dist')
    )
)

meal_sims.head()

meal_id,meal_id_sim,dist
i64,i64,f64
7005,7005,0.0
7005,9500150,0.521314
7005,9500039,0.494033
7005,899,0.416651
7005,950005,0.531856


## raw POS

In [9]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';', has_header=False, skip_rows=1)
    # df.columns = np.arange(df.shape[1], dtype=str)
    raw.append(df)


pos = pl.concat(raw)
pos.head()

column_1,column_2,column_3,column_4,column_5,column_6,column_7
str,str,str,str,str,i64,str
"""2.1.2023""","""10:31""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",1,"""0,9"""
"""2.1.2023""","""10:32""","""600 Chemicum""","""Kala""","""Kalapuikot tillikermaviilikast""",1,"""1,04"""
"""2.1.2023""","""10:32""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",1,"""0,9"""
"""2.1.2023""","""10:35""","""600 Chemicum""","""Kala""","""Kalapuikot tillikermaviilikast""",1,"""1,04"""
"""2.1.2023""","""10:36""","""600 Chemicum""","""Liha""","""Uunimakkaraa,sinappikastiketta""",2,"""1,8"""


In [10]:
# Rename columns
pos.columns = ['date', 'time', 'restaurant', 'meal_type', 'meal', 'pcs', 'co2']



# Convert pcs
pos = pos.filter((pl.col('pcs').is_not_null()) & (pl.col('pcs') >= 0))


# Map restaurant name
names_restaurant = {
    '600 Chemicum': 'che', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'exa', #'exactum'
    '570 Viikuna': 'vik',
}
pos = pos.with_columns(pl.col('restaurant').replace_strict(names_restaurant))


# Process date
pos = (
    pos
    .with_columns(
        (pl.col('date') + " " + pl.col('time')).str.to_datetime("%d.%m.%Y %H:%M").alias('datetime')
    )
    .drop('date', 'time')
)


# Process meal
pos = (
    pos

    # Remove trailing spaces
    .with_columns(
        pl.col('meal').str.strip_chars(' ')
    )

    
    # Get meal_id
    .join(meal_names, on='meal', how='left')

    
    # Remove entries having no `meal_id`
    .filter(pl.col('meal_id').is_not_null())
)



# Aggregate by date and supplement serving duration per day
pos = (
    pos
    .group_by('restaurant', pl.col('datetime').dt.date().alias('date'), 'meal_id')
    .agg(
        pl.col('pcs').sum(),
        pl.col('datetime').min().alias('time_start'),
        pl.col('datetime').max().alias('time_end'),
    )
    .with_columns(
        ((pl.col('time_end') - pl.col('time_start')).dt.total_minutes() / 60.).alias('serving_duration')
    )
    .join(
        dim_opentime.select('restaurant', 'working_duration'),
        on='restaurant',
        how='left'
    )
    .with_columns(
        (pl.col('serving_duration') / pl.col('working_duration')).alias('serving_percent')
    )
)



# Remove redundant columns
pos = pos.drop('time_start', 'time_end', 'serving_duration', 'working_duration')


# Add meal_type
meal_types = meals.select(
    'meal_id',
    pl.col('meal_type_1').alias('meal_type')
)
pos = pos.join(meal_types, on='meal_id', how='left')



# Ignore buffet
pos = pos.filter(pl.col('meal_type') != pl.lit('buffet'))



# Get top K meals with highest sale per day-restaurant
THETA = 4
pos = (
    pos
    .with_columns(
        pl.col('pcs') + pl.Series(np.random.rand(len(pos))) * 1e-3
    )
    .with_columns(
        pl.col('pcs').rank(method="dense", descending=True).over(['date', 'restaurant'], order_by='pcs').alias('rank')
    )
    .filter(pl.col('rank') <= THETA)
    .drop('rank')
)


pos.head()

restaurant,date,meal_id,pcs,serving_percent,meal_type
str,date,i64,f64,f64,str
"""che""",2024-01-08,1317,324.000066,1.062963,"""meat"""
"""phy""",2024-06-04,1454,14.000832,0.566667,"""vegetarian"""
"""exa""",2024-04-24,9500106,11.000817,0.366667,"""chicken"""
"""vik""",2023-04-06,9500116,28.000319,0.947619,"""chicken"""
"""exa""",2023-04-25,6938,120.000314,0.944444,"""meat"""


## dim `meals`

In [11]:
meals = pos.select('meal_id', 'restaurant', 'meal_type').unique()
meals.head()

meal_id,restaurant,meal_type
i64,str,str
9500168,"""che""","""vegan"""
22010,"""exa""","""vegan"""
9500094,"""che""","""fish"""
6088,"""exa""","""vegan"""
9500071,"""che""","""vegan"""


# Prepare data

## Refer every meals to top K highest sales meals per restaurant and meal_type

In [12]:
CUTOFF_DATE = pl.lit("2024-09-01", dtype=pl.Date)
pos_train = pos.filter(pl.col('date') < CUTOFF_DATE)

In [13]:
K = 14
topK = (
    pos_train
    .group_by('restaurant', 'meal_type', 'meal_id')
    .agg(pl.col('pcs').sum())
    .with_columns(
        pl.col('pcs').rank(descending=True).over('restaurant', 'meal_type').alias('rank')
    )
    .filter(pl.col('rank') <= K)
    .drop('rank', 'pcs')
)


# Add embedding
topK = topK.join(meal_embds, on='meal_id')



topK_grouped = (
    topK
    .select('restaurant', 'meal_type', 'meal_id')
    .group_by('restaurant', 'meal_type')
    .agg(
        pl.concat_list(pl.col('meal_id')).flatten()
    )
)
pos = (
    pos

    # Filter records related to topK and pos
    .join(
        topK_grouped,
        on=['restaurant', 'meal_type'],
        how='left'
    )
    .rename({'meal_id_right': 'meal_id_sim'})
)

pos.head()

restaurant,date,meal_id,pcs,serving_percent,meal_type,meal_id_sim
str,date,i64,f64,f64,str,list[i64]
"""che""",2024-01-08,1317,324.000066,1.062963,"""meat""","[9500029, 9500105, … 6945]"
"""phy""",2024-06-04,1454,14.000832,0.566667,"""vegetarian""","[1445, 1454, … 1450]"
"""exa""",2024-04-24,9500106,11.000817,0.366667,"""chicken""","[1292, 7586, … 9500153]"
"""vik""",2023-04-06,9500116,28.000319,0.947619,"""chicken""","[1292, 6851, … 9500153]"
"""exa""",2023-04-25,6938,120.000314,0.944444,"""meat""","[9500029, 9500030, … 8007]"


In [14]:
meal_embds = topK.select("meal_id", "embedding").unique()
meal_embds.head()

meal_id,embedding
i64,list[f64]
9500056,"[0.084854, 0.053983, … -0.014415]"
1445,"[0.021412, 0.005981, … 0.01204]"
1011,"[-0.038036, -0.032152, … 0.008614]"
7004,"[0.031372, 0.016187, … 0.024495]"
9500086,"[0.110537, 0.021696, … 0.03727]"


## Prepare encoders

In [15]:
# Encode meal_id
enc_meal_id = OrdinalEncoder()
enc_meal_id.fit(meal_embds.select('meal_id'))


# Encode restaurant
enc_restaurant = OrdinalEncoder()
enc_restaurant.fit(pos_train.select('restaurant'))


# Encode meal_type
enc_meal_type = OrdinalEncoder()
enc_meal_type.fit(pos_train.select('meal_type'))


# Scale pcs
scaler_pcs = MinMaxScaler((-1, 1))
scaler_pcs.fit(pos_train.select('pcs'))

MinMaxScaler(feature_range=(-1, 1))

## Start sampling

In [16]:
pos_grouped = (
    pos
    .group_by('restaurant', 'date')
    .agg(
        pl.concat_list('meal_id').flatten()
    )
)
pos_grouped.head()

restaurant,date,meal_id
str,date,list[i64]
"""che""",2024-09-09,"[9500070, 9042, … 7007]"
"""che""",2024-08-16,"[9500117, 6156, … 6673]"
"""che""",2024-02-13,"[9500147, 3056, … 9500146]"
"""phy""",2024-08-26,"[1513, 6137, … 1516]"
"""vik""",2024-04-11,"[927, 7006, … 7567]"


In [17]:
N = 6000

list_df = []
for _ in range(N):
    # Shuffle the meal order per day-restaurant
    pos_sample = (
        pos_grouped
        .select(
            'restaurant', 'date',
            pl.col('meal_id').list.sample(fraction=1, shuffle=True)
        )
        # .filter((pl.col('restaurant') == 'phy') & (pl.col('date') == pl.lit('2023-02-17', dtype=pl.Date)))
    )

    # Add row indices 
    pos_sample = (
        pos_sample
        .with_columns(
            pl.Series(
                np.random.randint(0, high=np.iinfo(np.uint64).max, size=len(pos_sample), dtype=np.uint64)
            ).alias('index')
        )
        .explode('meal_id')
    )


    # Get infor from table `pos`
    pos_sample = (
        pos_sample
        .join(
            pos.with_columns(pl.col('meal_id_sim').list.sample(n=1, shuffle=True).flatten()),
            on=['restaurant', 'date', 'meal_id'],
            how='left'
        )   
    )


    # Get similarity level
    pos_sample = (
        pos_sample
        .join(
            meal_sims,
            on=['meal_id', 'meal_id_sim'],
            how='left'
        )   
    )

    # Encode fields
    meal_id_sim_enc = enc_meal_id.transform(pos_sample.select(pl.col('meal_id_sim').alias('meal_id')))
    meal_type_enc = enc_meal_type.transform(pos_sample.select('meal_type'))
    
    pos_sample = (
        pos_sample
        .with_columns(
            pl.Series('meal_id_sim_enc', values=meal_id_sim_enc).arr.first().cast(pl.Int32),
            pl.Series('meal_type_enc', values=meal_type_enc).arr.first().cast(pl.Int32),
        )
    )

    # Scale
    pcs_scaled = scaler_pcs.transform(pos_sample.select('pcs'))
    pos_sample = (
        pos_sample
        .with_columns(
            pl.Series('pcs_scaled', values=pcs_scaled).arr.first().cast(pl.Float32),
        )
    )

    list_df.append(pos_sample)


pos_final = pl.concat(list_df)
pos_final.head()

restaurant,date,meal_id,index,pcs,serving_percent,meal_type,meal_id_sim,dist,meal_id_sim_enc,meal_type_enc,pcs_scaled
str,date,i64,u64,f64,f64,str,i64,f64,i32,i32,f32
"""che""",2024-09-09,7007,297616276942117957,32.000819,0.940741,"""vegan""",9500151,0.533036,140,3,-0.883892
"""che""",2024-09-09,9074,297616276942117957,352.000912,1.014815,"""chicken""",1751,0.352926,39,0,0.314609
"""che""",2024-09-09,9042,297616276942117957,121.000277,0.92963,"""vegan""",9500127,0.382149,131,3,-0.550561
"""che""",2024-09-09,9500070,297616276942117957,303.000088,1.011111,"""vegan""",9500068,0.243572,108,3,0.131086
"""che""",2024-08-16,9500117,4645879824722333004,149.00073,0.992593,"""meat""",9500030,0.565223,94,2,-0.445691


### Pivot

In [18]:
# restaurant = "che"
# date = pl.lit('2023-12-14', dtype=pl.Date)

cols_pivot = ['meal_id', 'serving_percent', 'meal_type_enc', 'meal_id_sim_enc', 'dist', 'pcs_scaled']

pos_final = (
    pos_final
    .with_columns(
        pl.col('date').rank(method='ordinal').over('index').alias('meal_rank')
    )
    .pivot(on='meal_rank', index=['index', 'date', 'restaurant'], values=cols_pivot)
    # .filter((pl.col('restaurant') == restaurant) & (pl.col('date') == date))
    # .filter(pl.col('meal_id_4').is_null())
)

pos_final.head()

index,date,restaurant,meal_id_1,meal_id_2,meal_id_3,meal_id_4,serving_percent_1,serving_percent_2,serving_percent_3,serving_percent_4,meal_type_enc_1,meal_type_enc_2,meal_type_enc_3,meal_type_enc_4,meal_id_sim_enc_1,meal_id_sim_enc_2,meal_id_sim_enc_3,meal_id_sim_enc_4,dist_1,dist_2,dist_3,dist_4,pcs_scaled_1,pcs_scaled_2,pcs_scaled_3,pcs_scaled_4
u64,date,str,i64,i64,i64,i64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f32,f32,f32,f32
297616276942117957,2024-09-09,"""che""",7007,9074,9042,9500070,0.940741,1.014815,0.92963,1.011111,3,0,3,3,140,39,131,108,0.533036,0.352926,0.382149,0.243572,-0.883892,0.314609,-0.550561,0.131086
4645879824722333004,2024-08-16,"""che""",9500117,6156,6673,7598,0.992593,0.951852,0.888889,0.985185,2,1,3,3,94,141,109,77,0.565223,0.456161,0.676154,0.605281,-0.445691,-0.483143,-0.460673,-0.838949
1429420781576332606,2024-02-13,"""che""",9500147,9500063,9500146,3056,0.988889,0.985185,0.003704,0.640741,1,3,1,3,1,52,24,109,0.639909,0.654993,0.378395,0.387614,0.048691,-0.191009,-0.827713,-0.505617
920832288553518737,2024-08-26,"""phy""",1513,1446,1516,6137,0.39,0.896667,0.56,0.566667,0,0,0,3,33,45,36,44,0.200889,0.359493,0.128673,0.539448,-0.962546,-0.96629,-0.96629,-0.973782
6009637599095154399,2024-04-11,"""vik""",927,7006,7567,6121,0.952381,0.952381,0.861905,0.619048,3,3,1,2,106,145,117,94,0.336008,0.532759,0.671771,0.500962,-0.666666,-0.617977,-0.441945,-0.857678


### Post processing

In [19]:
# Encoding: restaurant, date
pos_final = (
    pos_final
    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.month().alias('month'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )
    .drop('weekday', 'day', 'month')
)


restaurant_enc = enc_restaurant.transform(pos_final.select('restaurant'))
pos_final = (
    pos_final
    .with_columns(
        pl.Series('restaurant_enc', values=restaurant_enc).arr.first().cast(pl.Int32),
    )
)


# remove redundatn columns
pos_final = pos_final.drop(['index', 'restaurant', '^meal_id_.$'])


pos_final.head()

date,serving_percent_1,serving_percent_2,serving_percent_3,serving_percent_4,meal_type_enc_1,meal_type_enc_2,meal_type_enc_3,meal_type_enc_4,meal_id_sim_enc_1,meal_id_sim_enc_2,meal_id_sim_enc_3,meal_id_sim_enc_4,dist_1,dist_2,dist_3,dist_4,pcs_scaled_1,pcs_scaled_2,pcs_scaled_3,pcs_scaled_4,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,restaurant_enc
date,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,i32
2024-09-09,0.940741,1.014815,0.92963,1.011111,3,0,3,3,140,39,131,108,0.533036,0.352926,0.382149,0.243572,-0.883892,0.314609,-0.550561,0.131086,0.781831,0.62349,0.968077,-0.250653,-1.0,-1.8370e-16,0
2024-08-16,0.992593,0.951852,0.888889,0.985185,2,1,3,3,94,141,109,77,0.565223,0.456161,0.676154,0.605281,-0.445691,-0.483143,-0.460673,-0.838949,-0.974928,-0.222521,-0.101168,-0.994869,-0.866025,-0.5,0
2024-02-13,0.988889,0.985185,0.003704,0.640741,1,3,1,3,1,52,24,109,0.639909,0.654993,0.378395,0.387614,0.048691,-0.191009,-0.827713,-0.505617,0.974928,-0.222521,0.485302,-0.874347,0.866025,0.5,0
2024-08-26,0.39,0.896667,0.56,0.566667,0,0,0,3,33,45,36,44,0.200889,0.359493,0.128673,0.539448,-0.962546,-0.96629,-0.96629,-0.973782,0.781831,0.62349,-0.848644,0.528964,-0.866025,-0.5,2
2024-04-11,0.952381,0.952381,0.861905,0.619048,3,3,1,2,106,145,117,94,0.336008,0.532759,0.671771,0.500962,-0.666666,-0.617977,-0.441945,-0.857678,-0.433884,-0.900969,0.790776,-0.612106,0.866025,-0.5,3


## Split train-val

In [20]:
pos_train = pos_final.filter(pl.col('date') < CUTOFF_DATE).drop('date')
pos_val = pos_final.filter(pl.col('date') >= CUTOFF_DATE).drop('date')

pos_val.head()

serving_percent_1,serving_percent_2,serving_percent_3,serving_percent_4,meal_type_enc_1,meal_type_enc_2,meal_type_enc_3,meal_type_enc_4,meal_id_sim_enc_1,meal_id_sim_enc_2,meal_id_sim_enc_3,meal_id_sim_enc_4,dist_1,dist_2,dist_3,dist_4,pcs_scaled_1,pcs_scaled_2,pcs_scaled_3,pcs_scaled_4,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,restaurant_enc
f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,i32
0.940741,1.014815,0.92963,1.011111,3,0,3,3,140,39,131,108,0.533036,0.352926,0.382149,0.243572,-0.883892,0.314609,-0.550561,0.131086,0.781831,0.62349,0.968077,-0.250653,-1.0,-1.8370e-16,0
0.974074,0.974074,0.485185,0.996296,3,3,3,1,58,148,70,1,0.641376,0.598788,0.232092,0.586431,-0.340823,-0.614229,-0.786515,0.745319,0.974928,-0.222521,0.897805,-0.440394,-1.0,-1.8370e-16,0
0.966667,0.988889,0.977778,0.8,3,2,3,3,107,14,148,77,0.491364,0.400817,0.442057,0.652305,-0.479399,0.85019,-0.490636,-0.913858,0.781831,0.62349,-0.571268,0.820763,-0.866025,0.5,0
0.703333,0.696667,0.94,0.896667,2,0,0,4,34,45,90,29,0.0,0.0,0.519386,0.30965,-0.951309,-0.932582,-0.925092,-0.928836,0.974928,-0.222521,-0.968077,-0.250653,-0.866025,0.5,2
0.938889,0.966667,1.005556,0.588889,3,1,3,0,54,135,146,19,0.434808,0.492065,0.547489,0.366192,-0.808986,-0.456928,-0.382021,-0.883893,-0.433884,-0.900969,0.651372,-0.758758,-1.0,-1.8370e-16,1


# Save

In [21]:
path_dir = Path("data/inter/idea7")
path_dir.mkdir(exist_ok=True, parents=True)

In [22]:
path_train = path_dir / "train.parquet"
path_val = path_dir / "val.parquet"

pos_train.write_parquet(path_train)
pos_val.write_parquet(path_val)

In [23]:
path_scaler = path_dir / "pcs_scaler.gz"

joblib.dump(scaler_pcs, path_scaler)

['data/inter/idea7/pcs_scaler.gz']

In [24]:
meal_id_enc = enc_meal_id.transform(topK.select('meal_id'))
embedding = (
    topK
    .with_columns(
        pl.Series('meal_id', values=meal_id_enc).arr.first().cast(pl.Int32),
    )
    .group_by('meal_id')
    .first()
    .sort('meal_id')
    .get_column('embedding')
)
embds = np.stack(embedding.to_numpy())


path_embds = path_dir / "meal_embds.npy"
np.save(path_embds, embds)